Data Normalization & Augmentation

In [ ]:
import math
import random
from PIL import Image

def Data_Normalization_And_Augmentation(image_path, normalization_method):
    """Normalizes & Augments the Pixels in your Image, creating Several Prototypes"""
    base_img = Image.open(image_path).convert('RGB')
    orig_w, orig_h = base_img.size

    def pil_to_raw_matrix(pil_img):
        w, h = pil_img.size
        return [[[pil_img.getpixel((x, y))[c] for x in range(w)] for y in range(h)] for c in range(3)]

    original_image_tensor = pil_to_raw_matrix(base_img)

    normalized_image_tensor = [[[0.0 for _ in range(orig_w)] for _ in range(orig_h)] for _ in range(3)]
    means = [0.485, 0.456, 0.406]
    stds  = [0.229, 0.224, 0.225]

    for y in range(orig_h):
        for x in range(orig_w):
            r, g, b = base_img.getpixel((x, y))
            channels = [r, g, b]
            for c in range(3):
                scaled_p = channels[c] / 255.0 
                if normalization_method == "Min Max Scaling":
                    normalized_image_tensor[c][y][x] = scaled_p
                elif normalization_method == "Standard Deviation":
                    normalized_image_tensor[c][y][x] = (scaled_p - means[c]) / stds[c]

    crop_scale = random.uniform(0.8, 1.0)
    crop_w, crop_h = int(orig_w * crop_scale), int(orig_h * crop_scale)
    start_x = random.randint(0, orig_w - crop_w)
    start_y = random.randint(0, orig_h - crop_h)
    
    cropped_sub_img = base_img.crop((start_x, start_y, start_x + crop_w, start_y + crop_h))
    resized_crop_pil = Image.new('RGB', (244, 244))
    for y in range(244):
        for x in range(244):
            src_x = min(int(x * crop_w / 244), crop_w - 1)
            src_y = min(int(y * crop_h / 244), crop_h - 1)
            resized_crop_pil.putpixel((x, y), cropped_sub_img.getpixel((src_x, src_y)))
            
    resized_crop_tensor = pil_to_raw_matrix(resized_crop_pil)

    max_shift_x = int(orig_w * 0.1)
    max_shift_y = int(orig_h * 0.1)
    shift_x = random.randint(-max_shift_x, max_shift_x)
    shift_y = random.randint(-max_shift_y, max_shift_y)
    
    affine_pil = Image.new('RGB', (orig_w, orig_h), (0, 0, 0))  
    for y in range(orig_h):
        for x in range(orig_w):
            old_x = x - shift_x
            old_y = y - shift_y
            if 0 <= old_x < orig_w and 0 <= old_y < orig_h:
                affine_pil.putpixel((x, y), base_img.getpixel((old_x, old_y)))
                
    random_affine_tensor = pil_to_raw_matrix(affine_pil)

    angle_deg = random.uniform(-15, 15)
    angle_rad = math.radians(angle_deg)
    cos_a, sin_a = math.cos(angle_rad), math.sin(angle_rad)
    
    cx, cy = orig_w / 2.0, orig_h / 2.0  
    rotation_pil = Image.new('RGB', (orig_w, orig_h), (0, 0, 0))
    
    for y in range(orig_h):
        for x in range(orig_w):
            dx, dy = x - cx, y - cy
            old_x = int(cx + (dx * cos_a + dy * sin_a))
            old_y = int(cy + (-dx * sin_a + dy * cos_a))
            if 0 <= old_x < orig_w and 0 <= old_y < orig_h:
                rotation_pil.putpixel((x, y), base_img.getpixel((old_x, old_y)))
                
    random_rotation_tensor = pil_to_raw_matrix(rotation_pil)

    if random.random() > 0.5:
        flipped_pil = Image.new('RGB', (orig_w, orig_h))
        for y in range(orig_h):
            for x in range(orig_w):
                flipped_pil.putpixel((x, y), base_img.getpixel((orig_w - 1 - x, y)))
        random_horizontal_flipped_tensor = pil_to_raw_matrix(flipped_pil)
    else:
        random_horizontal_flipped_tensor = original_image_tensor

    return (
        original_image_tensor,
        normalized_image_tensor,
        resized_crop_tensor,
        random_affine_tensor,
        random_rotation_tensor,
        random_horizontal_flipped_tensor
    )


Batch Normalization & Dropout Layers

In [ ]:
import torch
import random

def Batch_Norm_And_Dropout(batch_activations, gamma, beta, network_instance, dropout_probability=0.20, training=True):
    
    """Normalizes the Layer Output as Batch Normalization & Kills off the Functionaliy of Certain Neurons as Dropout Layers"""
    batch_size, feature_size = batch_activations.shape
    bn_outputs = torch.zeros_like(batch_activations)
    eps = 1e-5  

    for col in range(feature_size):
        feature_sum = 0.0
        for row in range(batch_size):
            feature_sum += batch_activations[row, col].item()
        mean = feature_sum / batch_size
        
        variance_sum = 0.0
        for row in range(batch_size):
            variance_sum += (batch_activations[row, col].item() - mean) ** 2
        variance = variance_sum / batch_size
        
        for row in range(batch_size):
            raw_value = batch_activations[row, col].item()
            standardized_value = (raw_value - mean) / (variance + eps) ** 0.5
            
            bn_outputs[row, col] = (gamma[col].item() * standardized_value) + beta[col].item()

    if training:
        scale_factor = 1.0 / (1.0 - dropout_probability)
        
        for attribute_name in dir(network_instance):
            attribute_value = getattr(network_instance, attribute_name)
            
            if isinstance(attribute_value, torch.Tensor) and attribute_value.requires_grad:
                tensor_matrix = attribute_value
                b_size, num_neurons = tensor_matrix.shape
                
                pruned_matrix = torch.zeros_like(tensor_matrix)
                
                for row in range(b_size):
                    for col in range(num_neurons):
                        random_roll = random.random()
                        
                        if random_roll < dropout_probability:
                            pruned_matrix[row, col] = 0.0
                        else:
                            raw_val = tensor_matrix[row, col].item()
                            pruned_matrix[row, col] = raw_val * scale_factor
                
                setattr(network_instance, attribute_name, pruned_matrix)

    return bn_outputs

class PropertyRegressionModel:
    def __init__(self):
        self.layer1_neurons = torch.tensor([[10.0, 5.0, 15.0, 20.0]], requires_grad=True)

my_model = PropertyRegressionModel()

raw_layer_outputs = torch.tensor([
    [2.5, 4.0, 1.2, 8.5],
    [1.5, 6.0, 3.8, 11.5]
])

learned_gamma = torch.tensor([1.0, 1.0, 1.0, 1.0])
learned_beta  = torch.tensor([0.0, 0.0, 0.0, 0.0])

print("--- Before Pipeline Run ---")
print("Model Layer 1 Neurons :\n", my_model.layer1_neurons)

normalized_output_matrix = Batch_Norm_And_Dropout(
    batch_activations=raw_layer_outputs,
    gamma=learned_gamma,
    beta=learned_beta,
    network_instance=my_model,
    dropout_probability=0.25,
    training=True
)

print("\n--- After Pipeline Run ---")
print("Returned Normalized Matrix :\n", normalized_output_matrix)
print("\nModel Layer 1 Neurons :\n", my_model.layer1_neurons)

Residual Connections

In [ ]:
import torch

def Inject_Residual_Connections(network_instance, original_inputs, safety_seed):
    """Injects Residual Connections in your Network through Inputs"""

    for attribute_name in dir(network_instance):
        attribute_value = getattr(network_instance, attribute_name)
        
        if isinstance(attribute_value, torch.Tensor) and attribute_value.requires_grad:
            layer_neurons = attribute_value
            
            batch_size, layer_feature_size = layer_neurons.shape
            _, input_feature_size = original_inputs.shape
            
            updated_layer_state = torch.zeros_like(layer_neurons)
            
            if layer_feature_size == input_feature_size:
                for row in range(batch_size):
                    for col in range(layer_feature_size):
                        processed_val = layer_neurons[row, col].item()
                        raw_input_val = original_inputs[row, col].item()
                        
                        updated_layer_state[row, col] = processed_val + raw_input_val
            
            else:
                torch.manual_seed(safety_seed)
                projection_weights = torch.randn(input_feature_size, layer_feature_size)
                
                for row in range(batch_size):
                    for target_col in range(layer_feature_size):
                        
                        projected_input_value = 0.0
                        for input_col in range(input_feature_size):
                            input_val = original_inputs[row, input_col].item()
                            weight_val = projection_weights[input_col, target_col].item()
                            projected_input_value += input_val * weight_val
                        
                        processed_val = layer_neurons[row, target_col].item()
                        updated_layer_state[row, target_col] = processed_val + projected_input_value
            
            setattr(network_instance, attribute_name, updated_layer_state)

class DeepVisionNetwork:
    def __init__(self):
        self.layer1_neurons = torch.tensor([[10.0, 5.0, 15.0, 20.0, 8.0]], requires_grad=True)
        
        self.layer2_neurons = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]], requires_grad=True)

my_vision_model = DeepVisionNetwork()
raw_image_inputs = torch.tensor([[2.5, 3.0, 2.0, 22.0, 45.0]])

print("--- Before Residual Connection Injection ---")
print("Layer 1 Neurons ( 5 nodes ) :\n", my_vision_model.layer1_neurons)
print("Layer 2 Neurons ( 10 nodes ) :\n", my_vision_model.layer2_neurons)

Inject_Residual_Connections(network_instance=my_vision_model, original_inputs=raw_image_inputs, safety_seed=2)

print("\n--- After Residual Connection Injections ---")
print("Layer 1 Neurons ( Direct Addition Applied ) :\n", my_vision_model.layer1_neurons)
print("Layer 2 Neurons ( Projection Arithmetic Applied) :\n", my_vision_model.layer2_neurons)


Gradient Clipping, Weight Decay & Learning Rate Scheduling

In [ ]:
import torch

def Optimize_And_Schedule_Network(network_instance, initial_lr, current_epoch, step_size=10, gamma=0.1, weight_decay=1e-4, clip_value=1.0):
    """Integrates a System of Learning Rate Scheduling, Gradient Clipping & Weight Decay"""

    decay_exponent = current_epoch // step_size
    current_lr = initial_lr * (gamma ** decay_exponent)

    for attribute_name in dir(network_instance):
        attribute_value = getattr(network_instance, attribute_name)
        
        if isinstance(attribute_value, torch.Tensor) and attribute_value.requires_grad:
            param_tensor = attribute_value
            grad_tensor = param_tensor.grad
            
            if grad_tensor is None:
                continue
                
            updated_param_tensor = torch.zeros_like(param_tensor)
            
            if len(param_tensor.shape) == 2:
                rows, cols = param_tensor.shape
                for r in range(rows):
                    for c in range(cols):
                        raw_grad = grad_tensor[r, c].item()
                        raw_param = param_tensor[r, c].item()
                        
                        clipped_grad = max(-clip_value, min(clip_value, raw_grad))
                        
                        decayed_param = raw_param - (current_lr * weight_decay * raw_param)

                        updated_param_tensor[r, c] = decayed_param - (current_lr * clipped_grad)
            
            elif len(param_tensor.shape) == 1:
                length = param_tensor.shape[0]
                for i in range(length):
                    raw_grad = grad_tensor[i].item()
                    raw_param = param_tensor[i].item()
                    
                    clipped_grad = max(-clip_value, min(clip_value, raw_grad))
                    
                    decayed_param = raw_param - (current_lr * weight_decay * raw_param)
                    
                    updated_param_tensor[i] = decayed_param - (current_lr * clipped_grad)

            with torch.no_grad():
                param_tensor.copy_(updated_param_tensor)
                
    return current_lr

class DeepRegressionModel:
    def __init__(self):
        self.weight_matrix = torch.tensor([[1.5, -2.0], [0.5, 3.1]], requires_grad=True)
        self.weight_matrix.grad = torch.tensor([[50.0, -0.01], [12.4, -45.0]])

my_model_instance = DeepRegressionModel()
active_epoch = 12       
print("--- Before Global Optimization Pass ---")
print("Raw Parameter Weights :\n", my_model_instance.weight_matrix)
print("Pending Massive Gradients :\n", my_model_instance.weight_matrix.grad)

effective_lr = Optimize_And_Schedule_Network(
    network_instance=my_model_instance,
    initial_lr=0.1,
    current_epoch=active_epoch,
    step_size=10,   
    gamma=0.5,     
    weight_decay=1e-4,
    clip_value=1.0  
)

print("\n--- After Global Optimization Pass ---")
print(f"Operational Learning Rate applied ( Decayed ) : {effective_lr}")
print("Final Optimized Weights ( Clipped, Decayed, and Baseline Shifted ) :\n", my_model_instance.weight_matrix)


Ultimate Script

In [ ]:
import torch
import torch.nn as nn

class CustomConvolutionalArchitecture(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.flatten_features = 32 * 56 * 56
        
        self.fc1 = nn.Linear(in_features=self.flatten_features, out_features=128)
        self.relu3 = nn.ReLU()
        self.output_layer = nn.Linear(in_features=128, out_features=num_classes)
        self.conv1_neurons = None
        self.conv2_neurons = None
        self.fc1_neurons = None
        self.output_neurons = None

    def forward(self, x):
        out = self.conv1(x)
        out = self.relu1(out)
        out = self.pool1(out)
        self.conv1_neurons = out.clone().detach().requires_grad_(True)
        
        out = self.conv2(out)
        out = self.relu2(out)
        out = self.pool2(out)
        self.conv2_neurons = out.clone().detach().requires_grad_(True)
        
        out = out.view(out.size(0), -1)
        
        out = self.fc1(out)
        out = self.relu3(out)
        self.fc1_neurons = out.clone().detach().requires_grad_(True)
        
        logits = self.output_layer(out)
        self.output_neurons = logits.clone().detach().requires_grad_(True)
        
        return logits

if __name__ == "__main__":
    
    (original_tensor, 
     normalized_tensor, 
     resized_crop_tensor, 
     affine_tensor, 
     rotation_tensor, 
     flipped_tensor) = Data_Normalization_And_Augmentation("input_image.png", normalization_method="Standard Deviation")
    
    print(f"Successfully Created 6 Image Prototypes")

    cnn_model = CustomConvolutionalArchitecture(num_classes=10)
    print("Successfully Created a Convolutional Architectural Network")
    input_batch = torch.tensor([normalized_tensor], dtype=torch.float32)
    model_predictions = cnn_model(input_batch)

    mock_activations = torch.randn(1, 128)
    bn_gamma = torch.ones(128)
    bn_beta  = torch.zeros(128)
    
    normalized_batch_features = Batch_Norm_And_Dropout(
        batch_activations=mock_activations,
        gamma=bn_gamma,
        beta=bn_beta,
        network_instance=cnn_model,
        dropout_probability=0.30, 
        training=True
    )
    
    print("Successfully Integrated Batch Normalization & Dropout Layers in your Network")
    highway_inputs = torch.randn(1, 128)
    
    Inject_Residual_Connections(network_instance=cnn_model, original_inputs=highway_inputs)
    print("Successfully Injected Residual Connections into your Network")

    cnn_model.conv1.weight.grad = torch.randn_like(cnn_model.conv1.weight) * 250.0  
    cnn_model.conv1.bias.grad = torch.tensor([55.4] * 16)
    
    active_lr = Optimize_And_Schedule_Network(
        network_instance=cnn_model,
        initial_lr=0.1,
        current_epoch=25,
        step_size=10,
        gamma=0.5,        
        weight_decay=1e-4, 
        clip_value=1.0     
    )
    
    print(f"Succesfully Integrated Learning Rate Scheduling, Weight Decay & Gradient Clipping your Network, Learning Rate = {active_lr}")